##### Bronze Ingestion

Purpose

- Read ArcGIS REST API
- Preserve raw JSON
- Generate Run ID
- Write Bronze JSON

Output:

/Volumes/assignment/university/bronze/university_chapters/[run_id]

In [0]:
from datetime import datetime
import uuid

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_id += "_" + str(uuid.uuid4())[:8]

print(run_id)

20260722_031302_11b5b2b6


In [0]:
bronze_path = f"/Volumes/assignment/university/bronze/university_chapters/{run_id}"

dbutils.fs.mkdirs(bronze_path)

True

In [0]:
display(
    dbutils.fs.ls("/Volumes/assignment/university/bronze/")
)

path,name,size,modificationTime
dbfs:/Volumes/assignment/university/bronze/university_chapters/,university_chapters/,0,1784654862000


In [0]:
import requests

url = "https://services2.arcgis.com/5I7u4SJE1vUr79JC/arcgis/rest/services/UniversityChapters_Public/FeatureServer/0/query"

params = {
    "where": "State IN ('CA','OR','WA')",
    "outFields": "*",
    "returnGeometry": "true",
    "f": "json"
}

response = requests.get(url, params=params)

data = response.json()

In [0]:
import json
from pyspark.sql import Row

rows = []

for feature in data["features"]:
    rows.append(
        Row(raw_json=json.dumps(feature))
    )

bronze_df = spark.createDataFrame(rows)
display(bronze_df)

raw_json
"{""attributes"": {""OBJECTID"": 19, ""University_Chapter"": ""California Polytechnic State University"", ""City"": ""San Luis Obispo"", ""State"": ""CA"", ""ChapterID"": ""CA-0355"", ""MEVR_RD"": ""Derek Swindall""}, ""geometry"": {""x"": -120.66319100299995, ""y"": 35.274309145000075}}"
"{""attributes"": {""OBJECTID"": 26, ""University_Chapter"": ""Chico State University"", ""City"": ""Chico"", ""State"": ""CA"", ""ChapterID"": ""CA-0300"", ""MEVR_RD"": ""Brian Henderson""}, ""geometry"": {""x"": -121.83546324499997, ""y"": 39.73998176200007}}"
"{""attributes"": {""OBJECTID"": 1757, ""University_Chapter"": ""Fresno State"", ""City"": ""Fresno"", ""State"": ""CA"", ""ChapterID"": ""CA-0362"", ""MEVR_RD"": ""Garrett Williams""}, ""geometry"": {""x"": -119.73959481924653, ""y"": 36.82354541564933}}"


In [0]:
bronze_df.write \
    .mode("overwrite") \
    .json(bronze_path)

In [0]:
from pyspark.sql.functions import col, schema_of_json

# Infer schema from the first JSON string
json_schema = schema_of_json(
    bronze_df.select("raw_json").first()[0]
)

In [0]:
from pyspark.sql.functions import from_json

parsed_df = bronze_df.withColumn(
    "json_data",
    from_json(col("raw_json"), json_schema)
)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StructType

def flatten_df(df):
    """
    Recursively flattens all StructType columns in a DataFrame.
    """
    complex_fields = [
        (field.name, field.dataType)
        for field in df.schema.fields
        if isinstance(field.dataType, StructType)
    ]

    while complex_fields:
        col_name, _ = complex_fields.pop(0)

        expanded_cols = [
            col(f"{col_name}.{nested.name}").alias(nested.name)
            for nested in df.schema[col_name].dataType.fields
        ]

        other_cols = [
            col(c) for c in df.columns if c != col_name
        ]

        df = df.select(*other_cols, *expanded_cols)

        complex_fields = [
            (field.name, field.dataType)
            for field in df.schema.fields
            if isinstance(field.dataType, StructType)
        ]

    return df

In [0]:
from pyspark.sql.functions import from_json, col

parsed_df = bronze_df.withColumn(
    "json_data",
    from_json(col("raw_json"), json_schema)
)

flattened_df = flatten_df(parsed_df.select("json_data.*"))

display(flattened_df)

ChapterID,City,MEVR_RD,OBJECTID,State,University_Chapter,x,y
CA-0355,San Luis Obispo,Derek Swindall,19,CA,California Polytechnic State University,-120.66319100299995,35.274309145000075
CA-0300,Chico,Brian Henderson,26,CA,Chico State University,-121.83546324499997,39.73998176200007
CA-0362,Fresno,Garrett Williams,1757,CA,Fresno State,-119.73959481924653,36.82354541564933
